# SI26 Week 6 — Code Switching NLP Dataset
**Code Saviours SI-26 | Uswa Fatima**

This notebook builds a labelled Roman Urdu–English code-switching dataset:
191 mixed-language sentences (180 naturalistic constructed examples + 11
real messages collected from WhatsApp, redacted of personal info), each
word tagged as `URD`, `ENG`, or `MIX`.

**Pipeline:**
1. Load labelled sentence data
2. Flatten to a word-level CSV (`dataset.csv`)
3. Inspect label distribution and sanity-check
4. Export for upload to HuggingFace Datasets


In [5]:
import pandas as pd

data = [
    {
        'sentence': 'Yaar tension mat lo and the deadline is tomorrow',
        'words': ['Yaar', 'tension', 'mat', 'lo', 'and', 'the', 'deadline', 'is', 'tomorrow'],
        'labels': ['URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Ye traffic bilkul insane hai lekin I overslept again today',
        'words': ['Ye', 'traffic', 'bilkul', 'insane', 'hai', 'lekin', 'I', 'overslept', 'again', 'today'],
        'labels': ['URD', 'ENG', 'URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Kal ka din bohot acha tha aur had 3 meetings back to back',
        'words': ['Kal', 'ka', 'din', 'bohot', 'acha', 'tha', 'aur', 'had', '3', 'meetings', 'back', 'to', 'back'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Sab kuch theek chal raha hai lekin I need a break honestly',
        'words': ['Sab', 'kuch', 'theek', 'chal', 'raha', 'hai', 'lekin', 'I', 'need', 'a', 'break', 'honestly'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Kal raat neend hi nahi aayi phir can we reschedule the call',
        'words': ['Kal', 'raat', 'neend', 'hi', 'nahi', 'aayi', 'phir', 'can', 'we', 'reschedule', 'the', 'call'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Yaar kuch samajh nahi aa raha, still not prepared at all',
        'words': ['Yaar', 'kuch', 'samajh', 'nahi', 'aa', 'raha', 'still', 'not', 'prepared', 'at', 'all'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Ye traffic bilkul insane hai but this project is due Friday',
        'words': ['Ye', 'traffic', 'bilkul', 'insane', 'hai', 'but', 'this', 'project', 'is', 'due', 'Friday'],
        'labels': ['URD', 'ENG', 'URD', 'ENG', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Mujhe bilkul time nahi mila, I have zero motivation today',
        'words': ['Mujhe', 'bilkul', 'time', 'nahi', 'mila', 'I', 'have', 'zero', 'motivation', 'today'],
        'labels': ['URD', 'URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Ammi ne kaha ghar jaldi aa jao phir the deadline is tomorrow',
        'words': ['Ammi', 'ne', 'kaha', 'ghar', 'jaldi', 'aa', 'jao', 'phir', 'the', 'deadline', 'is', 'tomorrow'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Bhai kal mera presentation hai phir nothing is working properly',
        'words': ['Bhai', 'kal', 'mera', 'presentation', 'hai', 'phir', 'nothing', 'is', 'working', 'properly'],
        'labels': ['URD', 'URD', 'URD', 'MIX', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Kal raat neend hi nahi aayi lekin the results come out tomorrow',
        'words': ['Kal', 'raat', 'neend', 'hi', 'nahi', 'aayi', 'lekin', 'the', 'results', 'come', 'out', 'tomorrow'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Mujhe bilkul time nahi mila phir I'm not in the mood",
        'words': ['Mujhe', 'bilkul', 'time', 'nahi', 'mila', 'phir', "I'm", 'not', 'in', 'the', 'mood'],
        'labels': ['URD', 'URD', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Subha se sar dard ho raha hai aur I can't even right now",
        'words': ['Subha', 'se', 'sar', 'dard', 'ho', 'raha', 'hai', 'aur', 'I', "can't", 'even', 'right', 'now'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Yaar tension mat lo aur still not prepared at all',
        'words': ['Yaar', 'tension', 'mat', 'lo', 'aur', 'still', 'not', 'prepared', 'at', 'all'],
        'labels': ['URD', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Khana kha liya lekin this is so frustrating',
        'words': ['Khana', 'kha', 'liya', 'lekin', 'this', 'is', 'so', 'frustrating'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Ghar par sab log so gaye hain but the deadline is tomorrow',
        'words': ['Ghar', 'par', 'sab', 'log', 'so', 'gaye', 'hain', 'but', 'the', 'deadline', 'is', 'tomorrow'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Khana kha liya but I need a break honestly',
        'words': ['Khana', 'kha', 'liya', 'but', 'I', 'need', 'a', 'break', 'honestly'],
        'labels': ['URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Ye traffic bilkul insane hai and let's just get this done",
        'words': ['Ye', 'traffic', 'bilkul', 'insane', 'hai', 'and', "let's", 'just', 'get', 'this', 'done'],
        'labels': ['URD', 'ENG', 'URD', 'ENG', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Mujhe bilkul time nahi mila so honestly I don't care anymore",
        'words': ['Mujhe', 'bilkul', 'time', 'nahi', 'mila', 'so', 'honestly', 'I', "don't", 'care', 'anymore'],
        'labels': ['URD', 'URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Bahar bohot barish ho rahi hai and I totally forgot about it',
        'words': ['Bahar', 'bohot', 'barish', 'ho', 'rahi', 'hai', 'and', 'I', 'totally', 'forgot', 'about', 'it'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Kal raat neend hi nahi aayi so I have zero motivation today',
        'words': ['Kal', 'raat', 'neend', 'hi', 'nahi', 'aayi', 'so', 'I', 'have', 'zero', 'motivation', 'today'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Mera phone hang ho gaya hai phir this project is due Friday',
        'words': ['Mera', 'phone', 'hang', 'ho', 'gaya', 'hai', 'phir', 'this', 'project', 'is', 'due', 'Friday'],
        'labels': ['URD', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Kal raat neend hi nahi aayi but I'm not in the mood",
        'words': ['Kal', 'raat', 'neend', 'hi', 'nahi', 'aayi', 'but', "I'm", 'not', 'in', 'the', 'mood'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Bohot thak gaya hoon lekin I have zero motivation today',
        'words': ['Bohot', 'thak', 'gaya', 'hoon', 'lekin', 'I', 'have', 'zero', 'motivation', 'today'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Subha se sar dard ho raha hai so I have an assignment due',
        'words': ['Subha', 'se', 'sar', 'dard', 'ho', 'raha', 'hai', 'so', 'I', 'have', 'an', 'assignment', 'due'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Dil nahi kar raha kuch bhi karne ka and I need a break honestly',
        'words': ['Dil', 'nahi', 'kar', 'raha', 'kuch', 'bhi', 'karne', 'ka', 'and', 'I', 'need', 'a', 'break', 'honestly'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Mujhe bilkul time nahi mila and honestly I don't care anymore",
        'words': ['Mujhe', 'bilkul', 'time', 'nahi', 'mila', 'and', 'honestly', 'I', "don't", 'care', 'anymore'],
        'labels': ['URD', 'URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Mujhe neend aa rahi hai but I'm not in the mood",
        'words': ['Mujhe', 'neend', 'aa', 'rahi', 'hai', 'but', "I'm", 'not', 'in', 'the', 'mood'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Sab kuch theek chal raha hai so the wifi keeps disconnecting',
        'words': ['Sab', 'kuch', 'theek', 'chal', 'raha', 'hai', 'so', 'the', 'wifi', 'keeps', 'disconnecting'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Kal ka din bohot acha tha lekin let's just get this done",
        'words': ['Kal', 'ka', 'din', 'bohot', 'acha', 'tha', 'lekin', "let's", 'just', 'get', 'this', 'done'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Ammi ne kaha ghar jaldi aa jao aur I totally forgot about it',
        'words': ['Ammi', 'ne', 'kaha', 'ghar', 'jaldi', 'aa', 'jao', 'aur', 'I', 'totally', 'forgot', 'about', 'it'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Khana kha liya aur I'm literally so tired",
        'words': ['Khana', 'kha', 'liya', 'aur', "I'm", 'literally', 'so', 'tired'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Dil nahi kar raha kuch bhi karne ka phir honestly I don't care anymore",
        'words': ['Dil', 'nahi', 'kar', 'raha', 'kuch', 'bhi', 'karne', 'ka', 'phir', 'honestly', 'I', "don't", 'care', 'anymore'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Aaj mera mood nahi hai aur nothing is working properly',
        'words': ['Aaj', 'mera', 'mood', 'nahi', 'hai', 'aur', 'nothing', 'is', 'working', 'properly'],
        'labels': ['URD', 'URD', 'MIX', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Mujhe neend aa rahi hai, let's just get this done",
        'words': ['Mujhe', 'neend', 'aa', 'rahi', 'hai', "let's", 'just', 'get', 'this', 'done'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Bijli phir chali gayi and I have zero motivation today',
        'words': ['Bijli', 'phir', 'chali', 'gayi', 'and', 'I', 'have', 'zero', 'motivation', 'today'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Kal ka din bohot acha tha but I'm literally so tired",
        'words': ['Kal', 'ka', 'din', 'bohot', 'acha', 'tha', 'but', "I'm", 'literally', 'so', 'tired'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Aaj bohot garmi hai so I just want to sleep',
        'words': ['Aaj', 'bohot', 'garmi', 'hai', 'so', 'I', 'just', 'want', 'to', 'sleep'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Khana kha liya lekin I can't even right now",
        'words': ['Khana', 'kha', 'liya', 'lekin', 'I', "can't", 'even', 'right', 'now'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Dil nahi kar raha kuch bhi karne ka lekin I have zero motivation today',
        'words': ['Dil', 'nahi', 'kar', 'raha', 'kuch', 'bhi', 'karne', 'ka', 'lekin', 'I', 'have', 'zero', 'motivation', 'today'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Khana kha liya aur had 3 meetings back to back',
        'words': ['Khana', 'kha', 'liya', 'aur', 'had', '3', 'meetings', 'back', 'to', 'back'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Abhi tak reply nahi aya but I need to submit the form',
        'words': ['Abhi', 'tak', 'reply', 'nahi', 'aya', 'but', 'I', 'need', 'to', 'submit', 'the', 'form'],
        'labels': ['URD', 'URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Aaj bohot garmi hai lekin I was literally waiting for you',
        'words': ['Aaj', 'bohot', 'garmi', 'hai', 'lekin', 'I', 'was', 'literally', 'waiting', 'for', 'you'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Yaar kuch samajh nahi aa raha lekin I'm not in the mood",
        'words': ['Yaar', 'kuch', 'samajh', 'nahi', 'aa', 'raha', 'lekin', "I'm", 'not', 'in', 'the', 'mood'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Mera phone hang ho gaya hai aur I'm not in the mood",
        'words': ['Mera', 'phone', 'hang', 'ho', 'gaya', 'hai', 'aur', "I'm", 'not', 'in', 'the', 'mood'],
        'labels': ['URD', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Mujhe neend aa rahi hai aur honestly I don't care anymore",
        'words': ['Mujhe', 'neend', 'aa', 'rahi', 'hai', 'aur', 'honestly', 'I', "don't", 'care', 'anymore'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Kal ka din bohot acha tha, I just want to sleep',
        'words': ['Kal', 'ka', 'din', 'bohot', 'acha', 'tha', 'I', 'just', 'want', 'to', 'sleep'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Bohot thak gaya hoon and I'm literally so tired",
        'words': ['Bohot', 'thak', 'gaya', 'hoon', 'and', "I'm", 'literally', 'so', 'tired'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Mujhe bilkul time nahi mila but I overslept again today',
        'words': ['Mujhe', 'bilkul', 'time', 'nahi', 'mila', 'but', 'I', 'overslept', 'again', 'today'],
        'labels': ['URD', 'URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Mujhe bilkul time nahi mila but I'm literally so tired",
        'words': ['Mujhe', 'bilkul', 'time', 'nahi', 'mila', 'but', "I'm", 'literally', 'so', 'tired'],
        'labels': ['URD', 'URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Aaj mera mood nahi hai so the results come out tomorrow',
        'words': ['Aaj', 'mera', 'mood', 'nahi', 'hai', 'so', 'the', 'results', 'come', 'out', 'tomorrow'],
        'labels': ['URD', 'URD', 'MIX', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Subha se sar dard ho raha hai lekin this is so frustrating',
        'words': ['Subha', 'se', 'sar', 'dard', 'ho', 'raha', 'hai', 'lekin', 'this', 'is', 'so', 'frustrating'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Mujhe bilkul time nahi mila aur the results come out tomorrow',
        'words': ['Mujhe', 'bilkul', 'time', 'nahi', 'mila', 'aur', 'the', 'results', 'come', 'out', 'tomorrow'],
        'labels': ['URD', 'URD', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Yaar tension mat lo, I was literally waiting for you',
        'words': ['Yaar', 'tension', 'mat', 'lo', 'I', 'was', 'literally', 'waiting', 'for', 'you'],
        'labels': ['URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Aaj bohot garmi hai so I have zero motivation today',
        'words': ['Aaj', 'bohot', 'garmi', 'hai', 'so', 'I', 'have', 'zero', 'motivation', 'today'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Bohot thak gaya hoon so the wifi keeps disconnecting',
        'words': ['Bohot', 'thak', 'gaya', 'hoon', 'so', 'the', 'wifi', 'keeps', 'disconnecting'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Bahar bohot barish ho rahi hai so everyone is so stressed',
        'words': ['Bahar', 'bohot', 'barish', 'ho', 'rahi', 'hai', 'so', 'everyone', 'is', 'so', 'stressed'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Mujhe bilkul time nahi mila, everyone is so stressed',
        'words': ['Mujhe', 'bilkul', 'time', 'nahi', 'mila', 'everyone', 'is', 'so', 'stressed'],
        'labels': ['URD', 'URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Kal ka din bohot acha tha and I can't even right now",
        'words': ['Kal', 'ka', 'din', 'bohot', 'acha', 'tha', 'and', 'I', "can't", 'even', 'right', 'now'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Subha se sar dard ho raha hai but honestly I don't care anymore",
        'words': ['Subha', 'se', 'sar', 'dard', 'ho', 'raha', 'hai', 'but', 'honestly', 'I', "don't", 'care', 'anymore'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Bhai kal mera presentation hai but the results come out tomorrow',
        'words': ['Bhai', 'kal', 'mera', 'presentation', 'hai', 'but', 'the', 'results', 'come', 'out', 'tomorrow'],
        'labels': ['URD', 'URD', 'URD', 'MIX', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Abhi tak reply nahi aya but I overslept again today',
        'words': ['Abhi', 'tak', 'reply', 'nahi', 'aya', 'but', 'I', 'overslept', 'again', 'today'],
        'labels': ['URD', 'URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Sab kuch theek chal raha hai so the results come out tomorrow',
        'words': ['Sab', 'kuch', 'theek', 'chal', 'raha', 'hai', 'so', 'the', 'results', 'come', 'out', 'tomorrow'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Mujhe abhi ghar pohanchna hai, the results come out tomorrow',
        'words': ['Mujhe', 'abhi', 'ghar', 'pohanchna', 'hai', 'the', 'results', 'come', 'out', 'tomorrow'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Mujhe abhi ghar pohanchna hai, I need to submit the form',
        'words': ['Mujhe', 'abhi', 'ghar', 'pohanchna', 'hai', 'I', 'need', 'to', 'submit', 'the', 'form'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Yaar tension mat lo phir I need a break honestly',
        'words': ['Yaar', 'tension', 'mat', 'lo', 'phir', 'I', 'need', 'a', 'break', 'honestly'],
        'labels': ['URD', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Kal raat neend hi nahi aayi aur I need a break honestly',
        'words': ['Kal', 'raat', 'neend', 'hi', 'nahi', 'aayi', 'aur', 'I', 'need', 'a', 'break', 'honestly'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Mujhe bilkul time nahi mila and I have an assignment due',
        'words': ['Mujhe', 'bilkul', 'time', 'nahi', 'mila', 'and', 'I', 'have', 'an', 'assignment', 'due'],
        'labels': ['URD', 'URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Aaj mera mood nahi hai so this is so frustrating',
        'words': ['Aaj', 'mera', 'mood', 'nahi', 'hai', 'so', 'this', 'is', 'so', 'frustrating'],
        'labels': ['URD', 'URD', 'MIX', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Dil nahi kar raha kuch bhi karne ka, everyone is so stressed',
        'words': ['Dil', 'nahi', 'kar', 'raha', 'kuch', 'bhi', 'karne', 'ka', 'everyone', 'is', 'so', 'stressed'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Yaar tension mat lo so I have zero motivation today',
        'words': ['Yaar', 'tension', 'mat', 'lo', 'so', 'I', 'have', 'zero', 'motivation', 'today'],
        'labels': ['URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Kal se exams start ho rahe hain but everyone is so stressed',
        'words': ['Kal', 'se', 'exams', 'start', 'ho', 'rahe', 'hain', 'but', 'everyone', 'is', 'so', 'stressed'],
        'labels': ['URD', 'URD', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Abhi tak reply nahi aya aur still not prepared at all',
        'words': ['Abhi', 'tak', 'reply', 'nahi', 'aya', 'aur', 'still', 'not', 'prepared', 'at', 'all'],
        'labels': ['URD', 'URD', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Mujhe abhi ghar pohanchna hai but this is so frustrating',
        'words': ['Mujhe', 'abhi', 'ghar', 'pohanchna', 'hai', 'but', 'this', 'is', 'so', 'frustrating'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Ammi ne biryani banayi hai lekin the results come out tomorrow',
        'words': ['Ammi', 'ne', 'biryani', 'banayi', 'hai', 'lekin', 'the', 'results', 'come', 'out', 'tomorrow'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Bahar bohot barish ho rahi hai lekin still not prepared at all',
        'words': ['Bahar', 'bohot', 'barish', 'ho', 'rahi', 'hai', 'lekin', 'still', 'not', 'prepared', 'at', 'all'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Ye traffic bilkul insane hai aur let's just get this done",
        'words': ['Ye', 'traffic', 'bilkul', 'insane', 'hai', 'aur', "let's", 'just', 'get', 'this', 'done'],
        'labels': ['URD', 'ENG', 'URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Yaar tension mat lo but I'm running late again",
        'words': ['Yaar', 'tension', 'mat', 'lo', 'but', "I'm", 'running', 'late', 'again'],
        'labels': ['URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Ye traffic bilkul insane hai lekin had 3 meetings back to back',
        'words': ['Ye', 'traffic', 'bilkul', 'insane', 'hai', 'lekin', 'had', '3', 'meetings', 'back', 'to', 'back'],
        'labels': ['URD', 'ENG', 'URD', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Sab kuch theek chal raha hai, I need to submit the form',
        'words': ['Sab', 'kuch', 'theek', 'chal', 'raha', 'hai', 'I', 'need', 'to', 'submit', 'the', 'form'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Ye traffic bilkul insane hai so I have zero motivation today',
        'words': ['Ye', 'traffic', 'bilkul', 'insane', 'hai', 'so', 'I', 'have', 'zero', 'motivation', 'today'],
        'labels': ['URD', 'ENG', 'URD', 'ENG', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Mera phone hang ho gaya hai but I have zero motivation today',
        'words': ['Mera', 'phone', 'hang', 'ho', 'gaya', 'hai', 'but', 'I', 'have', 'zero', 'motivation', 'today'],
        'labels': ['URD', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Dil nahi kar raha kuch bhi karne ka aur I need a break honestly',
        'words': ['Dil', 'nahi', 'kar', 'raha', 'kuch', 'bhi', 'karne', 'ka', 'aur', 'I', 'need', 'a', 'break', 'honestly'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Subha se sar dard ho raha hai but still not prepared at all',
        'words': ['Subha', 'se', 'sar', 'dard', 'ho', 'raha', 'hai', 'but', 'still', 'not', 'prepared', 'at', 'all'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Aaj mera mood nahi hai lekin the results come out tomorrow',
        'words': ['Aaj', 'mera', 'mood', 'nahi', 'hai', 'lekin', 'the', 'results', 'come', 'out', 'tomorrow'],
        'labels': ['URD', 'URD', 'MIX', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Ammi ne kaha ghar jaldi aa jao but the deadline is tomorrow',
        'words': ['Ammi', 'ne', 'kaha', 'ghar', 'jaldi', 'aa', 'jao', 'but', 'the', 'deadline', 'is', 'tomorrow'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Bhai kal mera presentation hai phir I'm literally so tired",
        'words': ['Bhai', 'kal', 'mera', 'presentation', 'hai', 'phir', "I'm", 'literally', 'so', 'tired'],
        'labels': ['URD', 'URD', 'URD', 'MIX', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Ghar par sab log so gaye hain so I overslept again today',
        'words': ['Ghar', 'par', 'sab', 'log', 'so', 'gaye', 'hain', 'so', 'I', 'overslept', 'again', 'today'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Mera phone hang ho gaya hai so the results come out tomorrow',
        'words': ['Mera', 'phone', 'hang', 'ho', 'gaya', 'hai', 'so', 'the', 'results', 'come', 'out', 'tomorrow'],
        'labels': ['URD', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Bhai kal mera presentation hai aur can we reschedule the call',
        'words': ['Bhai', 'kal', 'mera', 'presentation', 'hai', 'aur', 'can', 'we', 'reschedule', 'the', 'call'],
        'labels': ['URD', 'URD', 'URD', 'MIX', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "honestly I don't care anymore so Ammi ne kaha ghar jaldi aa jao",
        'words': ['honestly', 'I', "don't", 'care', 'anymore', 'so', 'Ammi', 'ne', 'kaha', 'ghar', 'jaldi', 'aa', 'jao'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': "I can't even right now lekin Aaj bohot garmi hai",
        'words': ['I', "can't", 'even', 'right', 'now', 'lekin', 'Aaj', 'bohot', 'garmi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I was literally waiting for you, Ye traffic bilkul insane hai',
        'words': ['I', 'was', 'literally', 'waiting', 'for', 'you', 'Ye', 'traffic', 'bilkul', 'insane', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'ENG', 'URD', 'ENG', 'URD']
    },
    {
        'sentence': 'I need to submit the form lekin Subha se sar dard ho raha hai',
        'words': ['I', 'need', 'to', 'submit', 'the', 'form', 'lekin', 'Subha', 'se', 'sar', 'dard', 'ho', 'raha', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'the deadline is tomorrow lekin Aaj mera mood nahi hai',
        'words': ['the', 'deadline', 'is', 'tomorrow', 'lekin', 'Aaj', 'mera', 'mood', 'nahi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'MIX', 'URD', 'URD']
    },
    {
        'sentence': 'this project is due Friday but Yaar tension mat lo',
        'words': ['this', 'project', 'is', 'due', 'Friday', 'but', 'Yaar', 'tension', 'mat', 'lo'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': "it's not a big deal lekin Mujhe office jaldi jana hai",
        'words': ["it's", 'not', 'a', 'big', 'deal', 'lekin', 'Mujhe', 'office', 'jaldi', 'jana', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': "it's not a big deal and Ghar par sab log so gaye hain",
        'words': ["it's", 'not', 'a', 'big', 'deal', 'and', 'Ghar', 'par', 'sab', 'log', 'so', 'gaye', 'hain'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'the results come out tomorrow phir Kal raat neend hi nahi aayi',
        'words': ['the', 'results', 'come', 'out', 'tomorrow', 'phir', 'Kal', 'raat', 'neend', 'hi', 'nahi', 'aayi'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'everyone is so stressed phir Mera phone hang ho gaya hai',
        'words': ['everyone', 'is', 'so', 'stressed', 'phir', 'Mera', 'phone', 'hang', 'ho', 'gaya', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I totally forgot about it lekin Subha se sar dard ho raha hai',
        'words': ['I', 'totally', 'forgot', 'about', 'it', 'lekin', 'Subha', 'se', 'sar', 'dard', 'ho', 'raha', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I need to submit the form and Ammi ne biryani banayi hai',
        'words': ['I', 'need', 'to', 'submit', 'the', 'form', 'and', 'Ammi', 'ne', 'biryani', 'banayi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'the deadline is tomorrow so Ghar par sab log so gaye hain',
        'words': ['the', 'deadline', 'is', 'tomorrow', 'so', 'Ghar', 'par', 'sab', 'log', 'so', 'gaye', 'hain'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'the wifi keeps disconnecting but Bijli phir chali gayi',
        'words': ['the', 'wifi', 'keeps', 'disconnecting', 'but', 'Bijli', 'phir', 'chali', 'gayi'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': "let's just get this done phir Abhi tak reply nahi aya",
        'words': ["let's", 'just', 'get', 'this', 'done', 'phir', 'Abhi', 'tak', 'reply', 'nahi', 'aya'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': "I'm not in the mood lekin Aaj mera mood nahi hai",
        'words': ["I'm", 'not', 'in', 'the', 'mood', 'lekin', 'Aaj', 'mera', 'mood', 'nahi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'MIX', 'URD', 'URD']
    },
    {
        'sentence': 'the results come out tomorrow phir Mujhe abhi ghar pohanchna hai',
        'words': ['the', 'results', 'come', 'out', 'tomorrow', 'phir', 'Mujhe', 'abhi', 'ghar', 'pohanchna', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'nothing is working properly but Mera phone hang ho gaya hai',
        'words': ['nothing', 'is', 'working', 'properly', 'but', 'Mera', 'phone', 'hang', 'ho', 'gaya', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'ENG', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': "I can't even right now, Ghar par sab log so gaye hain",
        'words': ['I', "can't", 'even', 'right', 'now', 'Ghar', 'par', 'sab', 'log', 'so', 'gaye', 'hain'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'this is so frustrating so Mujhe bilkul time nahi mila',
        'words': ['this', 'is', 'so', 'frustrating', 'so', 'Mujhe', 'bilkul', 'time', 'nahi', 'mila'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': 'the deadline is tomorrow so Mujhe abhi ghar pohanchna hai',
        'words': ['the', 'deadline', 'is', 'tomorrow', 'so', 'Mujhe', 'abhi', 'ghar', 'pohanchna', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'had 3 meetings back to back, Ghar par sab log so gaye hain',
        'words': ['had', '3', 'meetings', 'back', 'to', 'back', 'Ghar', 'par', 'sab', 'log', 'so', 'gaye', 'hain'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I just want to sleep so Mujhe bilkul time nahi mila',
        'words': ['I', 'just', 'want', 'to', 'sleep', 'so', 'Mujhe', 'bilkul', 'time', 'nahi', 'mila'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': 'had 3 meetings back to back and Mujhe abhi ghar pohanchna hai',
        'words': ['had', '3', 'meetings', 'back', 'to', 'back', 'and', 'Mujhe', 'abhi', 'ghar', 'pohanchna', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': "it's not a big deal but Khana kha liya",
        'words': ["it's", 'not', 'a', 'big', 'deal', 'but', 'Khana', 'kha', 'liya'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I was literally waiting for you, Mujhe office jaldi jana hai',
        'words': ['I', 'was', 'literally', 'waiting', 'for', 'you', 'Mujhe', 'office', 'jaldi', 'jana', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'everyone is so stressed aur Ammi ne biryani banayi hai',
        'words': ['everyone', 'is', 'so', 'stressed', 'aur', 'Ammi', 'ne', 'biryani', 'banayi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'can we reschedule the call phir Yaar tension mat lo',
        'words': ['can', 'we', 'reschedule', 'the', 'call', 'phir', 'Yaar', 'tension', 'mat', 'lo'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': 'can we reschedule the call, Mera phone hang ho gaya hai',
        'words': ['can', 'we', 'reschedule', 'the', 'call', 'Mera', 'phone', 'hang', 'ho', 'gaya', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'ENG', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I overslept again today, Mujhe neend aa rahi hai',
        'words': ['I', 'overslept', 'again', 'today', 'Mujhe', 'neend', 'aa', 'rahi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I just want to sleep, Aaj mera mood nahi hai',
        'words': ['I', 'just', 'want', 'to', 'sleep', 'Aaj', 'mera', 'mood', 'nahi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'MIX', 'URD', 'URD']
    },
    {
        'sentence': 'nothing is working properly phir Ammi ne kaha ghar jaldi aa jao',
        'words': ['nothing', 'is', 'working', 'properly', 'phir', 'Ammi', 'ne', 'kaha', 'ghar', 'jaldi', 'aa', 'jao'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I overslept again today lekin Bijli phir chali gayi',
        'words': ['I', 'overslept', 'again', 'today', 'lekin', 'Bijli', 'phir', 'chali', 'gayi'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I have an assignment due but Aaj bohot garmi hai',
        'words': ['I', 'have', 'an', 'assignment', 'due', 'but', 'Aaj', 'bohot', 'garmi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': "I can't even right now but Yaar kuch samajh nahi aa raha",
        'words': ['I', "can't", 'even', 'right', 'now', 'but', 'Yaar', 'kuch', 'samajh', 'nahi', 'aa', 'raha'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I overslept again today but Ghar par sab log so gaye hain',
        'words': ['I', 'overslept', 'again', 'today', 'but', 'Ghar', 'par', 'sab', 'log', 'so', 'gaye', 'hain'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I overslept again today lekin Bohot thak gaya hoon',
        'words': ['I', 'overslept', 'again', 'today', 'lekin', 'Bohot', 'thak', 'gaya', 'hoon'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': "let's just get this done so Mujhe bilkul time nahi mila",
        'words': ["let's", 'just', 'get', 'this', 'done', 'so', 'Mujhe', 'bilkul', 'time', 'nahi', 'mila'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': 'I need to submit the form lekin Bohot thak gaya hoon',
        'words': ['I', 'need', 'to', 'submit', 'the', 'form', 'lekin', 'Bohot', 'thak', 'gaya', 'hoon'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': "I'm literally so tired aur Aaj mera mood nahi hai",
        'words': ["I'm", 'literally', 'so', 'tired', 'aur', 'Aaj', 'mera', 'mood', 'nahi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'MIX', 'URD', 'URD']
    },
    {
        'sentence': 'can we reschedule the call so Bohot thak gaya hoon',
        'words': ['can', 'we', 'reschedule', 'the', 'call', 'so', 'Bohot', 'thak', 'gaya', 'hoon'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'had 3 meetings back to back, Mujhe bilkul time nahi mila',
        'words': ['had', '3', 'meetings', 'back', 'to', 'back', 'Mujhe', 'bilkul', 'time', 'nahi', 'mila'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': 'can we reschedule the call lekin Kal se exams start ho rahe hain',
        'words': ['can', 'we', 'reschedule', 'the', 'call', 'lekin', 'Kal', 'se', 'exams', 'start', 'ho', 'rahe', 'hain'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'still not prepared at all, Ammi ne biryani banayi hai',
        'words': ['still', 'not', 'prepared', 'at', 'all', 'Ammi', 'ne', 'biryani', 'banayi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I have zero motivation today but Aaj bohot garmi hai',
        'words': ['I', 'have', 'zero', 'motivation', 'today', 'but', 'Aaj', 'bohot', 'garmi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'still not prepared at all lekin Sab kuch theek chal raha hai',
        'words': ['still', 'not', 'prepared', 'at', 'all', 'lekin', 'Sab', 'kuch', 'theek', 'chal', 'raha', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'the deadline is tomorrow phir Yaar tension mat lo',
        'words': ['the', 'deadline', 'is', 'tomorrow', 'phir', 'Yaar', 'tension', 'mat', 'lo'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': 'I have an assignment due phir Mujhe office jaldi jana hai',
        'words': ['I', 'have', 'an', 'assignment', 'due', 'phir', 'Mujhe', 'office', 'jaldi', 'jana', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'everyone is so stressed phir Mujhe bilkul time nahi mila',
        'words': ['everyone', 'is', 'so', 'stressed', 'phir', 'Mujhe', 'bilkul', 'time', 'nahi', 'mila'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': "it's not a big deal lekin Bohot thak gaya hoon",
        'words': ["it's", 'not', 'a', 'big', 'deal', 'lekin', 'Bohot', 'thak', 'gaya', 'hoon'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'can we reschedule the call phir Mujhe bilkul time nahi mila',
        'words': ['can', 'we', 'reschedule', 'the', 'call', 'phir', 'Mujhe', 'bilkul', 'time', 'nahi', 'mila'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': "I'm not in the mood phir Mujhe neend aa rahi hai",
        'words': ["I'm", 'not', 'in', 'the', 'mood', 'phir', 'Mujhe', 'neend', 'aa', 'rahi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': "I'm running late again aur Kal se exams start ho rahe hain",
        'words': ["I'm", 'running', 'late', 'again', 'aur', 'Kal', 'se', 'exams', 'start', 'ho', 'rahe', 'hain'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I have an assignment due aur Ghar par sab log so gaye hain',
        'words': ['I', 'have', 'an', 'assignment', 'due', 'aur', 'Ghar', 'par', 'sab', 'log', 'so', 'gaye', 'hain'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'nothing is working properly aur Mujhe office jaldi jana hai',
        'words': ['nothing', 'is', 'working', 'properly', 'aur', 'Mujhe', 'office', 'jaldi', 'jana', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I have an assignment due phir Kal ka din bohot acha tha',
        'words': ['I', 'have', 'an', 'assignment', 'due', 'phir', 'Kal', 'ka', 'din', 'bohot', 'acha', 'tha'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I have an assignment due, Yaar kuch samajh nahi aa raha',
        'words': ['I', 'have', 'an', 'assignment', 'due', 'Yaar', 'kuch', 'samajh', 'nahi', 'aa', 'raha'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'still not prepared at all aur Aaj bohot garmi hai',
        'words': ['still', 'not', 'prepared', 'at', 'all', 'aur', 'Aaj', 'bohot', 'garmi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': "honestly I don't care anymore but Yaar tension mat lo",
        'words': ['honestly', 'I', "don't", 'care', 'anymore', 'but', 'Yaar', 'tension', 'mat', 'lo'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': 'I overslept again today aur Aaj bohot garmi hai',
        'words': ['I', 'overslept', 'again', 'today', 'aur', 'Aaj', 'bohot', 'garmi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'this is so frustrating, Ghar par sab log so gaye hain',
        'words': ['this', 'is', 'so', 'frustrating', 'Ghar', 'par', 'sab', 'log', 'so', 'gaye', 'hain'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'still not prepared at all so Yaar kuch samajh nahi aa raha',
        'words': ['still', 'not', 'prepared', 'at', 'all', 'so', 'Yaar', 'kuch', 'samajh', 'nahi', 'aa', 'raha'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'this is so frustrating, Mera phone hang ho gaya hai',
        'words': ['this', 'is', 'so', 'frustrating', 'Mera', 'phone', 'hang', 'ho', 'gaya', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'ENG', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I have an assignment due so Sab kuch theek chal raha hai',
        'words': ['I', 'have', 'an', 'assignment', 'due', 'so', 'Sab', 'kuch', 'theek', 'chal', 'raha', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'the wifi keeps disconnecting phir Sab kuch theek chal raha hai',
        'words': ['the', 'wifi', 'keeps', 'disconnecting', 'phir', 'Sab', 'kuch', 'theek', 'chal', 'raha', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'I need a break honestly lekin Yaar tension mat lo',
        'words': ['I', 'need', 'a', 'break', 'honestly', 'lekin', 'Yaar', 'tension', 'mat', 'lo'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': 'I have an assignment due so Dil nahi kar raha kuch bhi karne ka',
        'words': ['I', 'have', 'an', 'assignment', 'due', 'so', 'Dil', 'nahi', 'kar', 'raha', 'kuch', 'bhi', 'karne', 'ka'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'had 3 meetings back to back phir Kal se exams start ho rahe hain',
        'words': ['had', '3', 'meetings', 'back', 'to', 'back', 'phir', 'Kal', 'se', 'exams', 'start', 'ho', 'rahe', 'hain'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'everyone is so stressed, Bahar bohot barish ho rahi hai',
        'words': ['everyone', 'is', 'so', 'stressed', 'Bahar', 'bohot', 'barish', 'ho', 'rahi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': "I'm running late again phir Aaj mera mood nahi hai",
        'words': ["I'm", 'running', 'late', 'again', 'phir', 'Aaj', 'mera', 'mood', 'nahi', 'hai'],
        'labels': ['ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'MIX', 'URD', 'URD']
    },
    {
        'sentence': 'Aaj ka din bohot busy tha, had 3 meetings back to back',
        'words': ['Aaj', 'ka', 'din', 'bohot', 'busy', 'tha', 'had', '3', 'meetings', 'back', 'to', 'back'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': "Yaar seriously I can't even right now, bohot thak gaya hoon",
        'words': ['Yaar', 'seriously', 'I', "can't", 'even', 'right', 'now', 'bohot', 'thak', 'gaya', 'hoon'],
        'labels': ['URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'Bhai kal mera presentation hai, still not prepared at all',
        'words': ['Bhai', 'kal', 'mera', 'presentation', 'hai', 'still', 'not', 'prepared', 'at', 'all'],
        'labels': ['URD', 'URD', 'URD', 'MIX', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Khana kha liya? I was literally waiting for you',
        'words': ['Khana', 'kha', 'liya', 'I', 'was', 'literally', 'waiting', 'for', 'you'],
        'labels': ['URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Yeh weather bilkul mausam kharab kar raha hai today',
        'words': ['Yeh', 'weather', 'bilkul', 'mausam', 'kharab', 'kar', 'raha', 'hai', 'today'],
        'labels': ['URD', 'MIX', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG']
    },
    {
        'sentence': 'I swear bhai mujhe koi idea nahi tha',
        'words': ['I', 'swear', 'bhai', 'mujhe', 'koi', 'idea', 'nahi', 'tha'],
        'labels': ['ENG', 'ENG', 'URD', 'URD', 'URD', 'MIX', 'URD', 'URD']
    },
    {
        'sentence': 'Class ke baad hum sab hangout karne ja rahe hain',
        'words': ['Class', 'ke', 'baad', 'hum', 'sab', 'hangout', 'karne', 'ja', 'rahe', 'hain'],
        'labels': ['MIX', 'URD', 'URD', 'URD', 'URD', 'ENG', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'Mujhe pata tha you would say that',
        'words': ['Mujhe', 'pata', 'tha', 'you', 'would', 'say', 'that'],
        'labels': ['URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Bas thori der mein reach kar jaunga',
        'words': ['Bas', 'thori', 'der', 'mein', 'reach', 'kar', 'jaunga'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': 'Honestly yaar mujhe kuch samajh nahi aa raha',
        'words': ['Honestly', 'yaar', 'mujhe', 'kuch', 'samajh', 'nahi', 'aa', 'raha'],
        'labels': ['ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'Kal exam tha, I totally blanked out',
        'words': ['Kal', 'exam', 'tha', 'I', 'totally', 'blanked', 'out'],
        'labels': ['URD', 'MIX', 'URD', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Everyone keh raha hai ke result kal aayega',
        'words': ['Everyone', 'keh', 'raha', 'hai', 'ke', 'result', 'kal', 'aayega'],
        'labels': ['ENG', 'URD', 'URD', 'URD', 'URD', 'MIX', 'URD', 'URD']
    },
    {
        'sentence': "Mujhe bhook lagi hai but I'm too lazy to cook",
        'words': ['Mujhe', 'bhook', 'lagi', 'hai', 'but', "I'm", 'too', 'lazy', 'to', 'cook'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'I think ye idea bilkul waste of time hai',
        'words': ['I', 'think', 'ye', 'idea', 'bilkul', 'waste', 'of', 'time', 'hai'],
        'labels': ['ENG', 'ENG', 'URD', 'MIX', 'URD', 'ENG', 'ENG', 'ENG', 'URD']
    },
    {
        'sentence': 'Chalo ab kaam pe focus karte hain',
        'words': ['Chalo', 'ab', 'kaam', 'pe', 'focus', 'karte', 'hain'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': 'She literally ghar se nikli hi nahi aaj',
        'words': ['She', 'literally', 'ghar', 'se', 'nikli', 'hi', 'nahi', 'aaj'],
        'labels': ['ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'Ab tak koi update nahi mila regarding the results',
        'words': ['Ab', 'tak', 'koi', 'update', 'nahi', 'mila', 'regarding', 'the', 'results'],
        'labels': ['URD', 'URD', 'URD', 'MIX', 'URD', 'URD', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Bro mujhe laga tu aa raha tha',
        'words': ['Bro', 'mujhe', 'laga', 'tu', 'aa', 'raha', 'tha'],
        'labels': ['ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'Itni garmi mein bhi she went for a walk',
        'words': ['Itni', 'garmi', 'mein', 'bhi', 'she', 'went', 'for', 'a', 'walk'],
        'labels': ['URD', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Kya scene hai aaj, are we still meeting up',
        'words': ['Kya', 'scene', 'hai', 'aaj', 'are', 'we', 'still', 'meeting', 'up'],
        'labels': ['URD', 'MIX', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'ENG', 'ENG']
    },
    {
        'sentence': 'Ye dr afia ki petition ha sign krdo',
        'words': ['Ye', 'dr', 'afia', 'ki', 'petition', 'ha', 'sign', 'krdo'],
        'labels': ['URD', 'ENG', 'URD', 'URD', 'ENG', 'URD', 'ENG', 'URD']
    },
    {
        'sentence': 'Ismein robot optional hai, aur software prototype ke through bhi strong FYP ban sakta hai',
        'words': ['Ismein', 'robot', 'optional', 'hai', 'aur', 'software', 'prototype', 'ke', 'through', 'bhi', 'strong', 'FYP', 'ban', 'sakta', 'hai'],
        'labels': ['URD', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'URD', 'ENG', 'URD', 'ENG', 'ENG', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'Main papers dekh kar ye practical gaps suggest karunga',
        'words': ['Main', 'papers', 'dekh', 'kar', 'ye', 'practical', 'gaps', 'suggest', 'karunga'],
        'labels': ['URD', 'ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'ENG', 'URD']
    },
    {
        'sentence': 'Agla required instrument predict kare',
        'words': ['Agla', 'required', 'instrument', 'predict', 'kare'],
        'labels': ['URD', 'ENG', 'ENG', 'ENG', 'URD']
    },
    {
        'sentence': 'Surgeon ke bolne se pehle recommendation de',
        'words': ['Surgeon', 'ke', 'bolne', 'se', 'pehle', 'recommendation', 'de'],
        'labels': ['ENG', 'URD', 'URD', 'URD', 'URD', 'ENG', 'URD']
    },
    {
        'sentence': 'Abhi free ni hon',
        'words': ['Abhi', 'free', 'ni', 'hon'],
        'labels': ['URD', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': 'Hardware based krna ha tum logon ne',
        'words': ['Hardware', 'based', 'krna', 'ha', 'tum', 'logon', 'ne'],
        'labels': ['ENG', 'ENG', 'URD', 'URD', 'URD', 'URD', 'URD']
    },
    {
        'sentence': 'Automata ke notes hn',
        'words': ['Automata', 'ke', 'notes', 'hn'],
        'labels': ['ENG', 'URD', 'ENG', 'URD']
    },
    {
        'sentence': 'Project ka kuch ya lab manual mn se',
        'words': ['Project', 'ka', 'kuch', 'ya', 'lab', 'manual', 'mn', 'se'],
        'labels': ['ENG', 'URD', 'URD', 'URD', 'ENG', 'ENG', 'URD', 'URD']
    },
    {
        'sentence': 'Streamlit pe to chalana ha bad mn usmn sai text nai show horaha',
        'words': ['Streamlit', 'pe', 'to', 'chalana', 'ha', 'bad', 'mn', 'usmn', 'sai', 'text', 'nai', 'show', 'horaha'],
        'labels': ['ENG', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'URD', 'ENG', 'URD', 'ENG', 'URD']
    },
    {
        'sentence': 'Accuracy kia aarae ha',
        'words': ['Accuracy', 'kia', 'aarae', 'ha'],
        'labels': ['ENG', 'URD', 'URD', 'URD']
    },
]

print(f'Total sentences collected: {len(data)}')


Total sentences collected: 191


## Convert to flat word-level CSV

In [ ]:
rows = []
for entry in data:
    for word, label in zip(entry['words'], entry['labels']):
        rows.append({
            'sentence': entry['sentence'],
            'word': word,
            'label': label
        })

df = pd.DataFrame(rows)
df.to_csv('dataset.csv', index=False, encoding='utf-8')

print(f'Dataset created: {len(df)} word entries')
print(f'Sentences: {df.sentence.nunique()}')
print(f'Label distribution:')
print(df.label.value_counts())


Dataset created: 2025 word entries
Sentences: 191
Label distribution:
label
URD    1002
ENG    1001
MIX      22
Name: count, dtype: int64


## Sanity checks

In [ ]:
assert df.sentence.nunique() >= 150, "Need at least 150 unique sentences"
assert set(df.label.unique()) <= {'URD', 'ENG', 'MIX'}, "Unexpected label found"
assert df.isnull().sum().sum() == 0, "Found missing values"

print("All checks passed ✅")
df.head(15)


All checks passed ✅


,sentence,word,label
0,Yaar tension mat lo and the deadline is tomorrow,Yaar,URD
1,Yaar tension mat lo and the deadline is tomorrow,tension,ENG
2,Yaar tension mat lo and the deadline is tomorrow,mat,URD
3,Yaar tension mat lo and the deadline is tomorrow,lo,URD
4,Yaar tension mat lo and the deadline is tomorrow,and,ENG
5,Yaar tension mat lo and the deadline is tomorrow,the,ENG
6,Yaar tension mat lo and the deadline is tomorrow,deadline,ENG
7,Yaar tension mat lo and the deadline is tomorrow,is,ENG
8,Yaar tension mat lo and the deadline is tomorrow,tomorrow,ENG
9,Ye traffic bilkul insane hai lekin I overslept...,Ye,URD
